Background and Problem Statement

TagPro is a free-to-play online 2-D capture the flag game in which players roll around a tiled map as paint-filled balls. Typically, the matches are played 4v4 with two players on each team on defense and the other two players on offense. While it can be a fun casual game, there is a staunch competitive community centered the game as well, not unlike the competitive scene around SSB Melee. Several regions (NA, EU, OCE) have professional leagues (MLTP, ELTP, OLTP respectively) where the best of the best battle it out to see who reigns supreme. There is a lot of raw data collected and recorded from each of these competitive matches, which are then reported either as is or used to construct more complex statistics. Going another step further, those statistics are then used to judge player performance, both directly by examining the leaders in each category and also by using comparison metrics that lump together several other statistics and weighting them appropriately.

This begs the question: "How do we choose the subcomponents of the player comparison metric in a reasonable manner, and subsequently weight them appropriately?" One reasonable approach to gathering information to this end is to use supervised learning algorithms to try to predict whether a player had a winning season based on their stats. We could define a winning season as having a positive cap differential; that is, the player's team scored more caps than they allowed their opponents to score against them. Figuring out which statistics are the most important for winning is crucial for players looking to improve their game, for general managers to decide which players they want to draft to their team, and for award voters to decide which players helped their teams the most.

Since the supervised investigation into that question was answered in the last investigation, we would like to now shift our attention to a related question and attempt to answer it using a bevy of unsupervised learning techniques coupled with the insight about which metrics are the most important from prior research. We would like to find players that are performing statistically far above or below the normal range in the statistical categories that have been determined to be the most relevant to winning using anomaly detection methods

We will look at data from the games of North American pro league, MLTP, spanning over two dozen seasons of matches. The data for each individual season is stored separately on a site called TagPro League, or TPL for short. We will need to scrape the raw data for each season and amalgamate everything, along with doing some non-trivial cleaning and preprocessing work.

[1] TagPro League. Available at: https://www.tagproleague.com/MLTP/stats (Accessed: April 22, 2023). 

In [35]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import warnings
import itertools
import plotly.express as px

A First Look at the Data

Let's examine the data we scraped from a single season of MLTP to get a better understanding of the underlying data structure and develop a plan for curating our final dataset.

In [36]:
warnings.filterwarnings('ignore', category = DeprecationWarning)
df = pd.read_csv('C:\\Users\\jshap\\OneDrive\\Desktop\\Stats\\MLTP_S10_majors_cumulative.tsv', delimiter = '\t', engine = 'python')
print(df)

           player team  score  minutes  pm  tags  pops  grabs  drops  hold  \
0         Altiger  NaN   1462      360   9   357   551    507    459  4078   
1      Blue Dream  NaN    143       40  -5    34    76     67     60   469   
2    Death Pickle  NaN   1579      400 -16   604   310    182    174  1461   
3         BigBird  NaN   1680      400  -4   550   284    170    158  1156   
4           Vials  NaN   1295      400 -16   286   645    594    542  4400   
..            ...  ...    ...      ...  ..   ...   ...    ...    ...   ...   
130       #Frozen  NaN     77       30 -19    31    66     57     54   227   
131       DUBSTEP  NaN     91       40 -28    28   102     83     77   353   
132     alchemist  NaN    105       40 -28    46    45     23     22   116   
133        Shabba  NaN     82       40 -28    29    74     60     57   276   
134  fenix funk 5  NaN    121       40 -28    59    53     26     26    82   

     ...  goodhandoff  goregrab  coregrab  gohandoff  cohandoff

We can see that there is no column to denote which season the data is from, so we will need to add that information into the master dataset as we weld the individual datasets together. By consulting with our Subject Matter Expert, we are also aware that there is an important shift that occurs starting in Season 20. Prior to that, most of the matches were played on teams' home servers, while in S20 and onwards all teams played on the Dallas-based server, resulting in much higher average ping for most players. To capture this critical rule change, we will also append a binary variable that tracks whether the season was played on Dallas or not.  

Building the Master Dataset

In [37]:
# Define the directory containing the TSV files
directory = 'C:\\Users\\jshap\\OneDrive\\Desktop\\Stats\\'

# Get a list of all TSV files in the directory
tsv_files = [os.path.join(directory, file) for file in os.listdir(directory) if file.endswith('.tsv')]

# Initialize an empty DataFrame to hold the data
MLTP_master_stats = pd.DataFrame()

# Loop over each TSV file and load it into a DataFrame
for i, location in enumerate(tsv_files):
    temp = pd.read_csv(location, sep='\t', header=0, na_values=[''], low_memory=False)
    season = i + 10
    Dallas = 0
    
    if season >= 20:
        Dallas = 1
    
    temp['Dallas'] = Dallas        
    temp['season'] = season
    MLTP_master_stats = MLTP_master_stats.append(temp, ignore_index=True)

# Convert necessary columns to factors
MLTP_master_stats['season'] = pd.Categorical(MLTP_master_stats['season'])
MLTP_master_stats['Dallas'] = pd.Categorical(MLTP_master_stats['Dallas'])

#Add ratio metrics (should NOT be per minute) 
MLTP_master_stats['kdr'] = MLTP_master_stats['tags'] / MLTP_master_stats['pops']
MLTP_master_stats['score_%'] = 100 * MLTP_master_stats['captures'] / MLTP_master_stats['grabs']

# Print the resulting DataFrame
print(MLTP_master_stats)
size = MLTP_master_stats.shape

            player team  score  minutes  pm  tags  pops  grabs  drops  hold  \
0          Altiger  NaN   1462      360   9   357   551    507    459  4078   
1       Blue Dream  NaN    143       40  -5    34    76     67     60   469   
2     Death Pickle  NaN   1579      400 -16   604   310    182    174  1461   
3          BigBird  NaN   1680      400  -4   550   284    170    158  1156   
4            Vials  NaN   1295      400 -16   286   645    594    542  4400   
...            ...  ...    ...      ...  ..   ...   ...    ...    ...   ...   
1307          Nati  TRT    159       50  10    65    50     40     39   186   
1308           bbb  ROJ    217       50   6    72    50     35     32   331   
1309          EIsa  NaN    165       50   3    34    86     83     74   472   
1310   TheEpicBlob  TGB    201       70   4    70    62     30     29   114   
1311       Pie Cat  NaN    196       50   8    54    69     63     54   426   

      ...  cohandoff  retinbase  quickret  saves  h

C:\Users\jshap\AppData\Local\Temp\ipykernel_19132\4141131219.py:21: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  MLTP_master_stats = MLTP_master_stats.append(temp, ignore_index=True)
C:\Users\jshap\AppData\Local\Temp\ipykernel_19132\4141131219.py:21: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  MLTP_master_stats = MLTP_master_stats.append(temp, ignore_index=True)
C:\Users\jshap\AppData\Local\Temp\ipykernel_19132\4141131219.py:21: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  MLTP_master_stats = MLTP_master_stats.append(temp, ignore_index=True)
C:\Users\jshap\AppData\Local\Temp\ipykernel_19132\4141131219.py:21: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pan

Now we have our full master dataset in all its glory! Let's investigate its properties a bit before getting ahead of ourselves with the data cleaning steps. We can see that we have 1312 observations, each containing 28 innate features and the four extra that we tacked on while creating the master dataset for a total of 32 features. The features 'Player' and 'team' are strings, 'Dallas' and 'season' are categorical features, 'Unnamed: 27' is a column full of all NaN values, and the other 27 features are various numeric integer statistical measures directly captured from the gameplay or calculated from directly captured data (such as Kill/Death Ratio and Scoring Percentage).

Data Cleaning

Let's programmatically check for missing values in each column for starters, in case the summary earlier didn't get the full picture:

In [38]:
for col in MLTP_master_stats.columns:
    temp = MLTP_master_stats[col].isna().sum()
    print(f"Missing values in feature '{col}': {temp} out of {size[0]}, or {round(100 * temp / size[0], 1)}% missing")

Missing values in feature 'player': 0 out of 1312, or 0.0% missing
Missing values in feature 'team': 1031 out of 1312, or 78.6% missing
Missing values in feature 'score': 0 out of 1312, or 0.0% missing
Missing values in feature 'minutes': 0 out of 1312, or 0.0% missing
Missing values in feature 'pm': 0 out of 1312, or 0.0% missing
Missing values in feature 'tags': 0 out of 1312, or 0.0% missing
Missing values in feature 'pops': 0 out of 1312, or 0.0% missing
Missing values in feature 'grabs': 0 out of 1312, or 0.0% missing
Missing values in feature 'drops': 0 out of 1312, or 0.0% missing
Missing values in feature 'hold': 0 out of 1312, or 0.0% missing
Missing values in feature 'captures': 0 out of 1312, or 0.0% missing
Missing values in feature 'prevent': 0 out of 1312, or 0.0% missing
Missing values in feature 'returns': 0 out of 1312, or 0.0% missing
Missing values in feature 'pups': 0 out of 1312, or 0.0% missing
Missing values in feature 'pupscomp': 0 out of 1312, or 0.0% missing
M

We can see that 'team' is missing almost 80% of its values, while 'Unnamed: 27' is entirely empty. There is no realistic way to impute values for either of these columns, so we will simply just drop them from the dataset entirely. Fortunately, almost every other column besides those two has exactly 0 missing values, so we are spared the effort of needing to impute anything at all. Note that there are a few missing values in the 'kdr' column, but we also know that 'kdr' is calculated directly from 'tags' divided by 'pops' and there is no missing data in either of those columns. What's going on here? A few of the players with a very low amount of minutes have 0 pops, which causes their KDR to became NaN due to a division by zero error. The same issue appears to plague the Scoring Percentage column as well, which would be expected if our hypothesis about why the data is missing is correct. We will circle back around to this after we filter out the players with too few minutes and we should see the missing values vanish.

In [39]:
MLTP_master_stats = MLTP_master_stats.drop(['team', 'Unnamed: 27'], axis = 1)
#print(MLTP_master_stats)

Another thing of importance to note from consultation with our SME is that different seasons had different numbers of weeks played, and some of the more recent seasons even had a different number of games and therefore minutes played each week. For a long time, the standard season length was 10 weeks with two 20 minute games a week for a total of 400 minutes played in a season. Season lengths started dropping down to around 7 weeks over time, and then in the past few seasons, the game format shifted from two 20 minute games to five 10 minute games a week. To handle this, we should temporally normalize all of these aggregation stats as per minute statistics instead.

In [40]:
MLTP_master_stats.columns

#Aggregation stats to be temporally normalized
agg_cols = ['score', 'pm', 'tags', 'pops', 'grabs', 'drops',
       'hold', 'captures', 'prevent', 'returns', 'pups', 'pupscomp',
       'flaccids', 'longholds', 'handoff', 'goodhandoff', 'goregrab',
       'coregrab', 'gohandoff', 'cohandoff', 'retinbase', 'quickret', 'saves',
       'holdagainst']

# Divide the columns by minutes played
MLTP_master_stats[agg_cols] = MLTP_master_stats[agg_cols].div(MLTP_master_stats['minutes'], axis = 0)

# Define a list of new column names
pm_cols = ['scorepm', 'successpm', 'tpm', 'poppm', 'gpm', 'dropspm',
       'hpm', 'cpm', 'ppm', 'rpm', 'puppm', 'pupscomppm',
       'flaccidspm', 'longholdspm', 'handoffpm', 'goodhandoffpm', 'goregrabpm',
       'coregrabpm', 'gohandoffpm', 'cohandoffpm', 'retinbasepm', 'quickretpm', 'savespm',
       'holdagainstpm']

# Create a dictionary of column name mappings
col_mapping = dict(zip(agg_cols, pm_cols))

# Rename the columns using the dictionary of column name mappings
MLTP_master_stats.rename(columns = col_mapping, inplace = True)

# Print the updated DataFrame
print(MLTP_master_stats)

            player   scorepm  minutes  successpm       tpm     poppm  \
0          Altiger  4.061111      360   0.025000  0.991667  1.530556   
1       Blue Dream  3.575000       40  -0.125000  0.850000  1.900000   
2     Death Pickle  3.947500      400  -0.040000  1.510000  0.775000   
3          BigBird  4.200000      400  -0.010000  1.375000  0.710000   
4            Vials  3.237500      400  -0.040000  0.715000  1.612500   
...            ...       ...      ...        ...       ...       ...   
1307          Nati  3.180000       50   0.200000  1.300000  1.000000   
1308           bbb  4.340000       50   0.120000  1.440000  1.000000   
1309          EIsa  3.300000       50   0.060000  0.680000  1.720000   
1310   TheEpicBlob  2.871429       70   0.057143  1.000000  0.885714   
1311       Pie Cat  3.920000       50   0.160000  1.080000  1.380000   

           gpm   dropspm        hpm       cpm  ...  gohandoffpm  cohandoffpm  \
0     1.408333  1.275000  11.327778  0.100000  ...     

On top of this, starting players will miss games and there will also be substitute players from the lower leagues filling in occasionally. We would like to exclude players that don't have many minutes played in a season, as they are likely either a minors player subbing or a majors player that got dropped for underperforming, either way not strong enough to be considered a majors level player. Additionally, a low number of minutes would make it more likely that their stats would be a huge outlier and throw off the model. Based on the recommendation of our SME, we will set this threshold as < 100 minutes played.  

In [41]:
# Filter out players with low minutes
MLTP_master_stats.drop(MLTP_master_stats[MLTP_master_stats['minutes'] < 100].index, inplace = True)
print(MLTP_master_stats)

            player   scorepm  minutes  successpm       tpm     poppm  \
0          Altiger  4.061111      360   0.025000  0.991667  1.530556   
2     Death Pickle  3.947500      400  -0.040000  1.510000  0.775000   
3          BigBird  4.200000      400  -0.010000  1.375000  0.710000   
4            Vials  3.237500      400  -0.040000  0.715000  1.612500   
5          Mr.Gone  3.925000      400  -0.040000  1.392500  1.027500   
...            ...       ...      ...        ...       ...       ...   
1301    BoldRoller  3.750000      268   0.070896  0.917910  1.727612   
1303         Prime  3.370000      200   0.100000  0.980000  1.150000   
1304         titan  3.070000      100  -0.040000  0.840000  1.760000   
1305  flaccid trip  3.887097      248  -0.129032  1.221774  1.120968   
1306            Ty  3.613445      238   0.096639  0.894958  1.218487   

           gpm   dropspm        hpm       cpm  ...  gohandoffpm  cohandoffpm  \
0     1.408333  1.275000  11.327778  0.100000  ...     

In [42]:
# Recheck dataframe
for col in MLTP_master_stats.columns:
    temp = MLTP_master_stats[col].isna().sum()
    print(f"Missing values in feature '{col}': {temp} out of {size[0]}, or {round(100 * temp / size[0], 1)}% missing")

Missing values in feature 'player': 0 out of 1312, or 0.0% missing
Missing values in feature 'scorepm': 0 out of 1312, or 0.0% missing
Missing values in feature 'minutes': 0 out of 1312, or 0.0% missing
Missing values in feature 'successpm': 0 out of 1312, or 0.0% missing
Missing values in feature 'tpm': 0 out of 1312, or 0.0% missing
Missing values in feature 'poppm': 0 out of 1312, or 0.0% missing
Missing values in feature 'gpm': 0 out of 1312, or 0.0% missing
Missing values in feature 'dropspm': 0 out of 1312, or 0.0% missing
Missing values in feature 'hpm': 0 out of 1312, or 0.0% missing
Missing values in feature 'cpm': 0 out of 1312, or 0.0% missing
Missing values in feature 'ppm': 0 out of 1312, or 0.0% missing
Missing values in feature 'rpm': 0 out of 1312, or 0.0% missing
Missing values in feature 'puppm': 0 out of 1312, or 0.0% missing
Missing values in feature 'pupscomppm': 0 out of 1312, or 0.0% missing
Missing values in feature 'flaccidspm': 0 out of 1312, or 0.0% missing
M

Now that all of the routine data cleaning has been executed, it's time to bring our SME knowledge into the fold. Based on the findings gleaned from previous research attempts, we are able to narrow down the list of statistics which are going to be the most interesting and relevant. Reducing the dimensionality of our data by excluding low importance statistical measures will cut out noise and eliminate some of the redundancy, which will make it easier for our models to find anomalies in critical areas. Let's do our routine EDA procdeures on the full feature set for a baseline look and then run the process again on the reduced set for comparison.

Another insight from previous investigation 